# 03b · 콜모고로프 방정식과 행렬 지수 (Kolmogorov Equations and the Matrix Exponential)

| 항목 | 내용 |
|---|---|
| 교재 지도 | Durrett 3e §4.2 (전이확률 계산), §4.3 (극한 거동) · Ross 12e §6.4 (전이확률 함수 $P_{ij}(t)$), §6.5 (극한 확률) · Norris §2.1 (Q-행렬과 지수), §2.8 (전진·후진 방정식), §3.6 (평형으로의 수렴) · Lawler 2e §3.2 (유한 상태공간, 고윳값 분해) |
| 선수 노트북 | 03a (생성원·질레스피·2상태 닫힌 형식), 01a (전이행렬과 채프먼–콜모고로프), 02d (검사 역설 — Recall Q4) |
| 예상 소요 | 90분 |
| 상태 | draft |

이 노트북의 단 하나의 아이디어: **연속시간에서 '전이행렬의 $n$제곱'을 대신하는 것은 행렬 지수 $e^{Qt}$다.** 03a에서 만든 CTMC의 $P(h)=I+Qh+o(h)$와 반군 성질 $P(s+t)=P(s)P(t)$를 미분하면 선형 상미분방정식(콜모고로프 방정식)이 나오고, 유한 상태에서 그 유일한 해가 $e^{Qt}$다. 시뮬레이션 없이 임의 시각의 분포를 계산할 수 있고, $t\to\infty$ 극한과 그 속도까지 $Q$의 고윳값이 말해 준다.

## 0. 준비 (Setup)

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import Markdown
from scipy.linalg import expm
from spkit import rng_for, fast, scale, setup_plots
from spkit.mc import mc_estimate, mc_proportion, compare_table, assert_close
from spkit.ctmc import is_generator, transition_matrix, stationary_ctmc, gillespie_paths, uniformize
from spkit.markov import tv_distance
from spkit.plots import plot_paths

SEED, rng = rng_for("03b")
setup_plots()
N_PATHS = scale(8_000)   # FAST(SPKIT_FAST=1) 모드에서는 1/10 크기
print(f"SEED={SEED}, fast={fast()}, N_PATHS={N_PATHS}")

## 1. Recall — 지난 노트북에서

책을 덮고 먼저 답을 적어 본 뒤 정답을 펼쳐 보세요.

**Q1.** (03a) 생성원 $Q$의 원소 $q_{ij}$ ($i\ne j$)와 $Q_{ii}$는 각각 무엇을 뜻하며 행합은 얼마인가?

<details><summary>정답</summary>

$q_{ij}$는 $i\to j$ 전이 비율(알람 시계의 속도), $Q_{ii}=-q_i=-\sum_{j\ne i}q_{ij}$ (총 탈출 비율에 음의 부호), 행합은 0.

</details>

**Q2.** (03a) 상태 $i$의 체류시간 분포와 다음 상태의 분포는? 둘은 어떤 관계인가?

<details><summary>정답</summary>

체류시간 $\sim\mathrm{Exp}(q_i)$, 다음 상태는 $j$일 확률 $q_{ij}/q_i$ (내재 점프 사슬), 그리고 둘은 **독립**이다 (02a 경쟁하는 시계 정리).

</details>

**Q3.** (03a) 2상태 사슬($0\to1$: $\alpha$, $1\to0$: $\beta$)의 $P_{01}(t)$와 수렴 속도는?

<details><summary>정답</summary>

$P_{01}(t)=\dfrac{\alpha}{\alpha+\beta}\big(1-e^{-(\alpha+\beta)t}\big)$; 극한 $\pi_1=\alpha/(\alpha+\beta)$로 지수 $\alpha+\beta$의 속도로 수렴한다.

</details>

**Q4.** (02d) 검사 역설: 비율 $\lambda$ 푸아송 과정에서 고정 시각 $t$를 덮는 구간 길이의 평균은 $t\to\infty$에서 얼마인가?

<details><summary>정답</summary>

$2/\lambda$ (일반 $t$에서는 $(2-e^{-\lambda t})/\lambda$). 평범한 간격의 평균 $1/\lambda$의 두 배 — 긴 구간이 고정 시각을 덮을 확률이 길이에 비례해 크기 때문이다.

</details>

## 2. 이론 — 내 말로 다시 쓰기

### 2.1 정의

- **전이확률 함수** $P(t)=(P_{ij}(t))_{i,j}$, $P_{ij}(t)=P(X_t=j\mid X_0=i)$. $P(0)=I$이고 각 행은 확률벡터(행합 1). **반군(semigroup) 성질**: $P(s+t)=P(s)P(t)$ — 채프먼–콜모고로프(Chapman–Kolmogorov) 방정식. 01a의 $P^{m+n}=P^mP^n$의 연속시간 판이다.
- **전진 방정식(forward equation)**: $P'(t)=P(t)Q$. 성분별로 $P'_{ij}(t)=\sum_k P_{ik}(t)\,q_{kj}$ — '$t$까지 $k$에 가 있다가 **마지막 순간**에 $k\to j$'.
- **후진 방정식(backward equation)**: $P'(t)=QP(t)$. 성분별로 $P'_{ij}(t)=\sum_k q_{ik}\,P_{kj}(t)$ — '**첫 순간**에 $i\to k$, 그 뒤 $t$ 동안 $k\to j$'.
- **행렬 지수(matrix exponential)** $e^{A}=\sum_{n\ge0}A^n/n!$. 모든 정방행렬에서 절대수렴한다 ($\|A^n\|\le\|A\|^n$). 성질: $\frac{d}{dt}e^{Qt}=Qe^{Qt}=e^{Qt}Q$, $e^{Q(s+t)}=e^{Qs}e^{Qt}$, $e^{Qt}=\lim_{n\to\infty}(I+Qt/n)^n$. 계산은 numpy 원소별 `np.exp`가 **아니라** `scipy.linalg.expm` (spkit: `transition_matrix(Q, t)`).
- **정상분포(stationary distribution)** (03c에서 정식화): $\pi Q=0$, $\sum_j\pi_j=1$이면 $\pi P(t)=\pi$ (모든 $t$). 기약 유한 CTMC에서 $P_{ij}(t)\to\pi_j$.
- **총변동 거리(total variation)** $\|P_{i\cdot}(t)-\pi\|_{TV}=\frac12\sum_j|P_{ij}(t)-\pi_j|$ (spkit `markov.tv_distance`). 수렴 속도를 재는 자.
- **$Q$의 스펙트럼**: 고윳값 $\lambda_1=0$ (우고유벡터 $\mathbf 1$: $Q\mathbf 1=0$은 행합 0과 같은 말), 나머지는 $\mathrm{Re}\,\lambda_k<0$; 기약이면 0은 단순근. 가역(세부 균형) 사슬이면 모든 고윳값이 실수, 아니면 복소쌍이 가능하다.
- **이 노트북의 예** (03a에서 재사용): 3상태 기계 사슬 $Q_3=\begin{pmatrix}-1.5&1.0&0.5\\1.0&-2.0&1.0\\2.0&0&-2.0\end{pmatrix}$ (0=정상, 1=성능저하, 2=고장; 고윳값 $0,\ -2.75\pm0.661i$, $\pi=(1/2,1/4,1/4)$)와 2상태 $Q_2=\begin{pmatrix}-1&1\\2&-2\end{pmatrix}$ (고윳값 $0,-3$, $\pi=(2/3,1/3)$).

### 2.2 정리 1 — 채프먼–콜모고로프 (Durrett 3e §4.2, Ross 12e §6.4)

> 모든 $s,t\ge0$에 대해 $P(s+t)=P(s)P(t)$, 즉 $P_{ij}(s+t)=\sum_k P_{ik}(s)\,P_{kj}(t)$.

**가정이 왜 필요한가**
- **마르코프 성질**: 시각 $s$의 상태 $k$가 주어지면 그 뒤 $t$ 동안의 전이가 과거($X_0=i$)와 무관해야 $P(X_{s+t}=j\mid X_s=k,X_0=i)=P(X_{s+t}=j\mid X_s=k)$로 줄어든다. 03a E3의 준마르코프 과정(상수 체류)에서는 이것이 깨지고 반군 성질도 깨진다 (E3).
- **시간 동질성**: $[s,s+t]$ 구간의 전이확률이 위치 $s$와 무관하게 $P(t)$여야 한다. 비동질이면 $P(s,s+t)$의 두 시각을 모두 써야 한다.

<details><summary>증명</summary>

시각 $s$의 상태로 전확률 공식을 쓰면
$$P_{ij}(s+t)=P(X_{s+t}=j\mid X_0=i)=\sum_k P(X_s=k\mid X_0=i)\,P(X_{s+t}=j\mid X_s=k,\,X_0=i).$$
첫 인수는 정의상 $P_{ik}(s)$. 둘째 인수는 마르코프 성질로 $P(X_{s+t}=j\mid X_s=k)$가 되고, 시간 동질성으로 $P(X_t=j\mid X_0=k)=P_{kj}(t)$가 된다. 따라서 $P_{ij}(s+t)=\sum_kP_{ik}(s)P_{kj}(t)$, 행렬로 $P(s+t)=P(s)P(t)$. $\square$

</details>

### 2.3 정리 2 — 콜모고로프 방정식과 유일해 $e^{Qt}$ (Norris §2.1, §2.8; Durrett 3e §4.2)

> 유한 상태공간에서 $P(h)=I+Qh+o(h)$ ($h\downarrow0$)이면 $P(t)$는 전진 방정식 $P'=PQ$와 후진 방정식 $P'=QP$를 **모두** 만족하며, $P(0)=I$인 해는 $P(t)=e^{Qt}$로 유일하다.

**가정이 왜 필요한가**
- **유한 상태공간**: $P(h)=I+Qh+o(h)$의 $o(h)$가 모든 성분에서 균일하고, 급수 $\sum(Qt)^n/n!$과 항별 미분·행렬곱의 극한 교환이 정당화된다. 무한 상태에서는 이 균일성이 깨져 **유일성**이 실패할 수 있다 — 폭발하는 사슬에서는 최소해 $P(t)$ 외에 다른 해가 있고 최소해의 행합이 1보다 작다 (Norris §2.8).
- **$P(0)=I$**: 초기조건 없이는 선형 ODE $M'=MQ$의 해가 $M(t)=Ce^{Qt}$로 한 가족이다.
- **$Q$가 상수(시간 동질)**: 시간 의존 $Q(t)$이면 해가 $e^{\int_0^tQ}$가 **아니다** — 서로 다른 시각의 $Q(t)$가 교환하지 않기 때문이다.

<details><summary>증명</summary>

**(도출, 전진)** 정리 1로 $P(t+h)=P(t)P(h)$이므로
$$P(t+h)-P(t)=P(t)\big(P(h)-I\big)=P(t)\big(Qh+o(h)\big).$$
$h$로 나누고 $h\downarrow0$: 우도함수 $P'(t)=P(t)Q$. (좌도함수도 같다: $P(t)-P(t-h)=P(t-h)(P(h)-I)$이고 $P(t-h)\to P(t)$ — 유한 상태에서 $P$는 연속이다.)

**(도출, 후진)** 반대로 $P(t+h)=P(h)P(t)$로 쓰면 $P(t+h)-P(t)=\big(P(h)-I\big)P(t)=\big(Qh+o(h)\big)P(t)$, 따라서 $P'(t)=QP(t)$.

**(해)** $E(t)=e^{Qt}=\sum_{n\ge0}Q^nt^n/n!$은 유한 차원에서 절대수렴하며 항별 미분이 허용된다:
$$E'(t)=\sum_{n\ge1}\frac{Q^nt^{n-1}}{(n-1)!}=Q\sum_{m\ge0}\frac{Q^mt^m}{m!}=QE(t)=E(t)Q,$$
$E(0)=I$. 즉 $e^{Qt}$는 두 방정식을 모두 만족한다.

**(유일성)** $M'=MQ$, $M(0)=I$인 임의의 해에 대해 $N(t)=M(t)e^{-Qt}$를 두면
$$N'(t)=M'(t)e^{-Qt}+M(t)\big(-Q\big)e^{-Qt}=M(t)Qe^{-Qt}-M(t)Qe^{-Qt}=0$$
($e^{-Qt}$는 $Q$의 급수이므로 $Q$와 교환한다). $N(0)=I$이므로 $N\equiv I$, 즉 $M(t)=e^{Qt}$ ($e^{-Qt}e^{Qt}=I$). 후진 방정식은 $N(t)=e^{-Qt}M(t)$로 같은 계산.

03a 정리 3(질레스피로 구성한 과정이 $P(h)=I+Qh+o(h)$를 만족)과 합치면: 알람 시계 구성 $(\Pi,q)$ $\Longleftrightarrow$ 전이확률 $e^{Qt}$. $\square$

</details>

### 2.4 정리 3 — 극한 정리, 주기 조건 없음 (Norris §3.6, Durrett 3e §4.3, Ross 12e §6.5)

> 유한 기약 CTMC에는 유일한 정상분포 $\pi$가 있고 모든 $i,j$에 대해 $P_{ij}(t)\to\pi_j$ ($t\to\infty$). DTMC(01e)와 달리 **비주기성 조건이 필요 없다.**

**가정이 왜 필요한가**
- **기약**: 닫힌 부분집합이 둘 이상이면 출발 상태에 따라 극한이 다르다 (정상분포도 유일하지 않다).
- **유한**: 무한 상태에서는 양재귀성(정상분포의 존재)이 추가로 필요하다 — M/M/1에서 $\rho\ge1$이면 $P_{ij}(t)\to0$으로 극한 **분포**가 없다 (03d).
- **비주기성이 필요 없는 이유**: $t>0$이면 기약 CTMC의 $P_{ij}(t)$는 **모두 양수**다 (한 번 $i\to j$ 경로가 있으면 그 점프들이 $[0,t]$ 안에 다 일어나고 나머지 시간엔 머무는 사건의 확률이 양수). DTMC의 주기적 반례(01e: $0\leftrightarrow1$ 왕복)는 $P^n$의 원소가 0과 양수를 번갈아 갖는 데서 오는데, 연속시간에는 '정확히 $n$스텝'이 없어 그런 현상이 없다. 03f에서 균일화 사슬 $I+Q/\lambda$가 자기 루프(양의 대각)를 가져 자동으로 비주기적임을 본다.

**증명 스케치.** $\lambda>\max_iq_i$를 잡고 $P=I+Q/\lambda$를 두면 $P$는 대각 원소가 양수인 기약 확률행렬, 즉 기약·비주기 DTMC의 전이행렬이다. 01e의 수렴 정리로 $P^n\to\mathbf 1\pi$ ($\pi P=\pi\iff\pi Q=0$). 한편 $P(t)=e^{Qt}=e^{\lambda t(P-I)}=e^{-\lambda t}e^{\lambda tP}=\sum_{n\ge0}e^{-\lambda t}\frac{(\lambda t)^n}{n!}P^n$ (03f 정리 1, 균일화)이고 푸아송$(\lambda t)$ 가중치는 $t\to\infty$에서 큰 $n$쪽으로 몰리므로 $P(t)\to\mathbf 1\pi$. 완전한 증명은 Norris §3.6 / Durrett 3e §4.3.

### 2.5 정리 4 — 스펙트럼과 수렴 속도 (Lawler 2e §3.2)

> $Q$가 대각화 가능하고 고윳값을 실수부 내림차순으로 $0=\lambda_1,\lambda_2,\dots,\lambda_n$ ($0>\mathrm{Re}\,\lambda_2\ge\mathrm{Re}\,\lambda_3\ge\cdots$)으로 늘어놓으면
> $$P(t)=\mathbf 1\pi+\sum_{k\ge2}e^{\lambda_kt}\,v_ku_k^{\top}$$
> ($v_k$: 우고유벡터, $u_k^\top$: 좌고유벡터, $u_k^\top v_l=\delta_{kl}$)이므로 $|P_{ij}(t)-\pi_j|\le Ce^{\mathrm{Re}\lambda_2\,t}$. 2상태 사슬에서는 정확히 $\|P_{0\cdot}(t)-\pi\|_{TV}=\dfrac{\alpha}{\alpha+\beta}e^{-(\alpha+\beta)t}$.

**가정이 왜 필요한가**
- **대각화 가능**: 조르당 블록이 있으면 $t^me^{\lambda t}$ 항이 생긴다 (지수 속도 자체는 같다).
- **복소 고윳값**이면 $e^{\mathrm{Re}\lambda t}\cos(\mathrm{Im}\lambda\,t+\phi)$ 진동이 섞여 $\log$ TV 곡선이 직선이 아니라 요철을 갖는다 — $Q_3$가 그 예다. 가역 사슬은 고윳값이 실수라 진동이 없다.

**증명 스케치.** $Q=V\Lambda V^{-1}$이면 $Q^n=V\Lambda^nV^{-1}$이므로 급수에서 $e^{Qt}=Ve^{\Lambda t}V^{-1}=\sum_ke^{\lambda_kt}v_ku_k^\top$. $k=1$ 항: $v_1=\mathbf 1$, $u_1^\top$은 $u_1^\top Q=0$인 좌고유벡터로 정규화 $u_1^\top\mathbf 1=1$을 주면 $u_1=\pi$, 즉 $v_1u_1^\top=\mathbf 1\pi$. 2상태 계산은 03a 정리 2에서 직접: $P_{00}(t)-\pi_0=\pi_1e^{-(\alpha+\beta)t}$, $P_{01}(t)-\pi_1=-\pi_1e^{-(\alpha+\beta)t}$이므로 TV는 그 절댓값 $\pi_1e^{-(\alpha+\beta)t}$. 가역 사슬(01f, 03c)의 고윳값이 실수인 이유는 $D^{1/2}QD^{-1/2}$ ($D=\mathrm{diag}(\pi)$)가 대칭행렬이기 때문이다.

### 2.6 직관

$e^{Qt}=\lim_n(I+Qt/n)^n$: 시간을 $n$조각으로 잘라 각 조각에서 '확률행렬 $I+Q\Delta t$'로 굴리는 극한이다 — 03a의 격자 근사가 $n\to\infty$에서 정확해진다는 뜻이고, 03f 균일화는 같은 아이디어를 극한 없이 정확하게 만든 것이다. 전진 방정식은 '**마지막 순간**에 무슨 일이 일어났나'($P(t)$ 뒤에 $Q$), 후진 방정식은 '**첫 순간**에 무슨 일이 일어났나'($Q$ 뒤에 $P(t)$)로 조건화한 것이며, 유한 상태에서는 두 답이 같다 ($e^{Qt}$가 $Q$와 교환). 수렴은 $\log$ TV 그림으로 본다: 2상태는 기울기 $-(\alpha+\beta)=-3$인 완벽한 직선, $Q_3$는 기울기 $\mathrm{Re}\lambda_2=-2.75$의 참조선 주위로 느리게 진동하는(주기 $2\pi/\mathrm{Im}\lambda_2\approx9.5$) 요철 있는 곡선 — 비가역 사슬의 복소 고윳값이 남긴 흔적이다.

### 2.7 함정(traps)

1. `np.exp(Q*t)`는 **원소별** 지수이고 틀렸다. 행렬 지수는 `scipy.linalg.expm(Q*t)` (급수 $\sum(Qt)^n/n!$). 4.2절에서 원소별 지수의 행합이 1이 아님을 직접 본다.
2. $P(t)$의 행합이 1인 것은 $Q$의 행합이 0에서 온다 ($Q\mathbf 1=0\Rightarrow e^{Qt}\mathbf 1=\mathbf 1$). $Q$의 부호나 대각을 잘못 넣으면 행합이 1에서 벗어난다 (E3).
3. CTMC에는 주기가 없다. '정상분포가 있어도 주기 때문에 수렴 안 함'은 DTMC 이야기(01e)이고, 기약 유한 CTMC는 항상 수렴한다.
4. 수렴 속도는 $|\lambda_2|$가 아니라 $\mathrm{Re}\,\lambda_2$(음수)가 정한다. 복소 고윳값이면 TV 곡선이 진동해 두 시각 사이의 기울기가 $\mathrm{Re}\lambda_2$와 다를 수 있다 (4.5절: $t=4\to6$ 구간 기울기 $\approx-2.92\ne-2.75$).
5. 유한 차분으로 $P'(t)$를 검증할 때 $h$를 너무 작게(1e-8 이하) 잡으면 반올림 오차가 지배한다. $h\approx10^{-4}$면 중앙차분 오차 $\sim10^{-9}$ (4.3절).
6. 질레스피 격자 점유율은 이항 비율이므로 SE는 $\sqrt{p(1-p)/n}$. 같은 경로에서 여러 $t$를 읽으면 시각 간 **상관**이 있지만 각 시각의 추정 자체는 편향이 없다.

✍️ 내 말로: (책을 덮고 여기에 핵심 정의·정리를 다시 써 보세요)

- (a) 전진/후진 방정식의 차이를 '첫 스텝 조건화 vs 마지막 스텝 조건화'로 한 문단 설명해 보세요.
- (b) 왜 CTMC에는 01e의 주기적 반례가 없는지 한 문장으로 써 보세요.
- (c) 정리 2의 유일성 증명에서 $N(t)=M(t)e^{-Qt}$의 도함수가 0이 되는 계산을 다시 해 보세요.

## 3. Predict — 코드를 돌리기 전에 예측

아래 셀의 `None`을 숫자로 바꿔 보세요. 정확할 필요는 없습니다 — 자릿수와 방향이 맞는지가 목적입니다. 힌트: $Q_3$의 정상분포는 $(1/2,1/4,1/4)$이고, 2.1절의 고윳값을 보면 $t=1$에서 $\pi$까지 얼마나 남았는지 가늠할 수 있다.

In [ ]:
predictions = {
    # Q3 기계가 정상(0)에서 출발해 t=1 에 고장(2) 상태일 확률 P_02(1) 은?
    "p02_at_1": None,
    # 같은 조건에서 t=1 에 여전히(또는 다시) 정상(0)일 확률 P_00(1) 은?
    "p00_at_1": None,
    # t=2 에서 P_0.(2) 와 pi=(0.5, 0.25, 0.25) 의 총변동 거리는 대략 얼마인가? (자릿수만)
    "tv_at_2": None,
    # 2상태 사슬(alpha=1, beta=2)이 0에서 출발할 때 t=1 에서 pi=(2/3, 1/3) 까지의 총변동 거리는?
    "tv_two_state_at_1": None,
    # Q3 의 0이 아닌 고윳값의 실수부(수렴 지수)는?
    "re_lambda2_Q3": None,
}

## 4. Simulate — 시뮬레이션

### 4.1 $P(t)=e^{Qt}$ 계산 (단계 1)

03a의 두 생성원을 다시 만들고, `transition_matrix(Q, t)` (= `scipy.linalg.expm(Q*t)`)로 몇 개의 $t$에서 $P(t)$를 계산한다. 확인할 것: $P(0)=I$, 모든 행합이 1, 그리고 $t$가 커질수록 세 행이 같은 벡터 $\pi$로 모인다.

In [ ]:
Q3 = np.array([[-1.5, 1.0, 0.5],
               [ 1.0, -2.0, 1.0],
               [ 2.0, 0.0, -2.0]])          # 0=정상, 1=성능저하, 2=고장
Q2 = np.array([[-1.0, 1.0],
               [ 2.0, -2.0]])               # alpha=1, beta=2
assert is_generator(Q3) and is_generator(Q2)
pi3, pi2 = stationary_ctmc(Q3), stationary_ctmc(Q2)
print(f"pi(Q3) = {pi3}, pi(Q2) = {pi2}")

for t in (0.0, 0.5, 1.0, 2.0, 3.0):
    Pt = transition_matrix(Q3, t)
    np.testing.assert_allclose(Pt.sum(axis=1), 1.0, atol=1e-12)     # 행합 1
    print(f"t={t:.1f}  P(t) =\n{Pt.round(6)}")
np.testing.assert_allclose(transition_matrix(Q3, 0.0), np.eye(3))   # P(0) = I
print("row sums = 1 and P(0) = I: ok")

In [ ]:
t_grid = np.linspace(0, 4, 200)
P_curve = np.array([transition_matrix(Q3, t) for t in t_grid])    # shape (200, 3, 3)

fig, ax = plt.subplots()
names = ["0 (up)", "1 (degraded)", "2 (down)"]
for j in range(3):
    ax.plot(t_grid, P_curve[:, 0, j], color=f"C{j}", label=f"P_0{j}(t) -> state {names[j]}")
    ax.axhline(pi3[j], color=f"C{j}", ls="--", lw=1)
ax.set_xlabel("t"); ax.set_ylabel("probability")
ax.set_title("Fig 1: Transition probabilities from state 0 (expm), pi_j dashed")
ax.legend()
plt.show()

$P_{00}(t)$는 1에서 $\pi_0=1/2$로 내려가고, $P_{02}(t)$는 0에서 출발해 $\pi_2=1/4$로 올라간다. 세 곡선은 $t\approx2$면 이미 점선과 구분되지 않는다 — $e^{-2.75\cdot2}\approx0.004$이기 때문이다.

### 4.2 반군·교환 성질과 $\lim(I+Qt/n)^n$ (단계 2)

정리 1, 2의 대수적 항등식은 부동소수점 오차 수준에서 **정확히** 성립해야 한다. 함께 함정 1(원소별 `np.exp`)의 행합도 본다.

In [ ]:
P1, P2, P3 = (transition_matrix(Q3, t) for t in (1.0, 2.0, 3.0))
err_semigroup = np.abs(P1 @ P2 - P3).max()          # P(1)P(2) = P(3)
err_commute = np.abs(P1 @ Q3 - Q3 @ P1).max()       # e^{Q} Q = Q e^{Q}
print(f"semigroup |P(1)P(2) - P(3)|_max = {err_semigroup:.1e}")
print(f"commute   |P(1)Q - QP(1)|_max   = {err_commute:.1e}")
assert err_semigroup < 1e-10 and err_commute < 1e-10

print(f"trap: np.exp(Q3) row sums = {np.exp(Q3).sum(axis=1).round(4)}  (elementwise exp is NOT P(1))")

print("\n|(I + Q/n)^n - expm(Q)|_max  (t = 1):")
for n in (10, 100, 1000, 10_000):
    approx = np.linalg.matrix_power(np.eye(3) + Q3 / n, n)
    print(f"  n = {n:>6d}: {np.abs(approx - P1).max():.3e}")

오차가 $n$에 정확히 반비례한다(1차 격자 근사). 03a의 시간 격자 시뮬레이션은 바로 이 $(I+Q\Delta t)^{n}$를 확률적으로 실행한 것이고, 그 이산화 편향이 $O(\Delta t)$인 이유가 여기 있다.

### 4.3 전진·후진 방정식의 유한 차분 검증 (단계 3)

$t=1$에서 중앙차분 $\dfrac{P(1+h)-P(1-h)}{2h}$를 $P(1)Q$ (전진), $QP(1)$ (후진)와 비교한다. 중앙차분 오차는 $O(h^2)$이지만 $h$가 아주 작아지면 $\epsilon_{\text{machine}}/h$ 반올림 오차가 커진다 (함정 5).

In [ ]:
h_grid = np.logspace(-1, -7, 7)
err_fwd, err_bwd = [], []
for h in h_grid:
    D = (transition_matrix(Q3, 1.0 + h) - transition_matrix(Q3, 1.0 - h)) / (2 * h)
    err_fwd.append(np.abs(D - P1 @ Q3).max())
    err_bwd.append(np.abs(D - Q3 @ P1).max())
err_fwd, err_bwd = np.array(err_fwd), np.array(err_bwd)
for h, ef, eb in zip(h_grid, err_fwd, err_bwd):
    print(f"h = {h:.0e}: forward err {ef:.2e}   backward err {eb:.2e}")

D4 = (transition_matrix(Q3, 1 + 1e-4) - transition_matrix(Q3, 1 - 1e-4)) / 2e-4
assert np.abs(D4 - P1 @ Q3).max() < 1e-6     # forward equation at h = 1e-4
assert np.abs(D4 - Q3 @ P1).max() < 1e-6     # backward equation
print("finite-difference check of P' = PQ = QP at t = 1: ok")

In [ ]:
fig, ax = plt.subplots()
ax.loglog(h_grid, err_fwd, "o-", label="|central diff - P(1) Q|  (forward)")
ax.loglog(h_grid, err_bwd, "s--", ms=4, label="|central diff - Q P(1)|  (backward)")
ax.loglog(h_grid, err_fwd[0] * (h_grid / h_grid[0]) ** 2, "k:", label="slope 2 reference (O(h^2))")
ax.set_xlabel("h"); ax.set_ylabel("max abs error")
ax.set_title("Fig 2: Finite-difference check of dP/dt = PQ at t = 1")
ax.legend()
plt.show()

$h=10^{-1}\to10^{-4}$ 구간은 기울기 2의 직선(절단 오차 $O(h^2)$), 그 아래는 반올림 바닥이다. 전진과 후진의 오차가 같은 것은 $P(1)Q=QP(1)$이기 때문 — 유한 상태에서 두 방정식은 같은 해를 준다.

### 4.4 질레스피 vs $e^{Qt}$ (단계 4)

이제 두 세계를 맞댄다: 03a의 질레스피 알고리즘(알람 시계 구성)으로 경로를 뽑아 시각 $t$의 상태 분포를 세면, 정리 2에 따라 $e^{Qt}$의 0행과 몬테카를로 오차 안에서 일치해야 한다.

In [ ]:
T_sim = 3.0
t_g, X = gillespie_paths(Q3, 0, T_sim, rng, N_PATHS, n_grid=301)
occ = np.stack([(X == j).mean(axis=0) for j in range(3)])      # (3, 301) 격자 점유율

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
plot_paths(t_g, X, ax=ax1, n_show=5, step=True, alpha=0.7)
ax1.set_yticks([0, 1, 2], ["0 up", "1 degraded", "2 down"])
ax1.set_xlabel("t"); ax1.set_title("Fig 3a: Five Gillespie paths of the machine chain")
P_sim = np.array([transition_matrix(Q3, t) for t in t_g])
for j in range(3):
    ax2.plot(t_g, P_sim[:, 0, j], color=f"C{j}", label=f"expm: P_0{j}(t)")
    ax2.plot(t_g[::10], occ[j, ::10], "o", color=f"C{j}", ms=4, alpha=0.8)
ax2.set_xlabel("t"); ax2.set_ylabel("fraction of paths in state j")
ax2.set_title(f"Fig 3b: Gillespie occupation (markers, n={N_PATHS}) vs expm rows (lines)")
ax2.legend()
plt.tight_layout(); plt.show()

In [ ]:
idx = int(np.argmin(np.abs(t_g - 1.0)))            # t = 1 격자 인덱스
est_p0 = mc_proportion(X[:, idx] == 0)
est_p1 = mc_proportion(X[:, idx] == 1)
est_p2 = mc_proportion(X[:, idx] == 2)
P1_row0 = transition_matrix(Q3, 1.0)[0]
for j, est in enumerate((est_p0, est_p1, est_p2)):
    print(f"P_0{j}(1)  Gillespie {est}   expm {P1_row0[j]:.6f}")

점(시뮬레이션)이 선(해석해) 위에 얹힌다. 각 시각의 SE는 $\sqrt{p(1-p)/n}$이고 (함정 6), 5절에서 $t=1$의 세 값을 4SE 기준으로 판정한다.

### 4.5 $\pi$로의 수렴과 그 속도 (단계 5)

정리 3, 4를 그림으로 본다. 먼저 두 생성원의 고윳값, 그리고 $\|P_{0\cdot}(t)-\pi\|_{TV}$를 $t\in[0,6]$에서 계산해 semilogy로 그린다.

In [ ]:
eig3, eig2 = np.linalg.eigvals(Q3), np.linalg.eigvals(Q2)
print("eigvals(Q3) =", eig3.round(4))
print("eigvals(Q2) =", eig2.round(4))
nonzero = eig3[np.abs(eig3) > 1e-9]                  # lambda_1 = 0 을 제외
re_lambda2 = float(nonzero.real.max())              # 가장 느린 모드 = 실수부가 가장 큰(0에 가까운) 고윳값
re_lambda2_char = np.trace(Q3) / 2                   # 특성다항식 lambda(lambda^2 + 5.5 lambda + 8): 켤레쌍의 실수부 = tr(Q3)/2 = -5.5/2
print(f"Re lambda_2(Q3) = {re_lambda2:.4f} (from eigvals) vs {re_lambda2_char:.4f} (characteristic polynomial)")

P_unif, lam_u = uniformize(Q3, lam=2.5)              # lam > max q_i = 2 이면 모든 대각이 양수
print(f"uniformized chain I + Q/lam (lam={lam_u}): diag = {np.diag(P_unif)} -> self-loops, aperiodic")

In [ ]:
t_tv = np.linspace(0, 6, 121)
tv3 = np.array([tv_distance(transition_matrix(Q3, t)[0], pi3) for t in t_tv])
tv2 = np.array([tv_distance(transition_matrix(Q2, t)[0], pi2) for t in t_tv])
tv2_exact = pi2[1] * np.exp(-3 * t_tv)                       # 정리 4: pi_1 e^{-(alpha+beta) t}

fig, ax = plt.subplots()
ax.semilogy(t_tv, tv3, "C0", lw=2, label="3-state Q3 (complex eigenvalues)")
ax.semilogy(t_tv, tv3[0] * np.exp(re_lambda2 * t_tv), "C0:", label=f"reference slope Re lambda_2 = {re_lambda2:.2f}")
ax.semilogy(t_tv, tv2, "C1", lw=2, label="2-state Q2 (alpha=1, beta=2)")
ax.semilogy(t_tv[::6], tv2_exact[::6], "C1o", ms=4, label="(1/3) exp(-3t) exact")
ax.set_ylim(1e-9, 1); ax.set_xlabel("t"); ax.set_ylabel("TV distance to pi from state 0")
ax.set_title("Fig 4: Convergence to pi: straight line (2-state) vs oscillating decay (3-state)")
ax.legend(fontsize=8)
plt.show()

In [ ]:
tv2_at_1 = tv_distance(transition_matrix(Q2, 1.0)[0], pi2)
tv2_at_1_exact = (1 / 3) * np.exp(-3.0)
np.testing.assert_allclose(tv2_at_1, tv2_at_1_exact, rtol=1e-8)
print(f"2-state TV(1) = {tv2_at_1:.10f} = (1/3)e^-3 = {tv2_at_1_exact:.10f}")

tv3_at_2 = tv_distance(transition_matrix(Q3, 2.0)[0], pi3)
tv3_at_4, tv3_at_6 = (tv_distance(transition_matrix(Q3, t)[0], pi3) for t in (4.0, 6.0))
print(f"3-state TV(2) = {tv3_at_2:.7f}, TV(4) = {tv3_at_4:.3e}, TV(6) = {tv3_at_6:.3e}")
print(f"log-slope of TV between t=4 and t=6: {(np.log(tv3_at_6) - np.log(tv3_at_4)) / 2:.3f}  (Re lambda_2 = {re_lambda2:.2f})")

# 정리 4의 스펙트럼 분해로 P(2)를 다시 조립해 expm 과 대조: V e^{Lambda t} V^{-1}
lam_v, V = np.linalg.eig(Q3)
P2_spec = (V @ np.diag(np.exp(lam_v * 2.0)) @ np.linalg.inv(V)).real
tv3_at_2_spec = tv_distance(P2_spec[0], pi3)
k1 = int(np.argmin(np.abs(lam_v)))                      # lambda = 0 항
rank1 = np.outer(V[:, k1], np.linalg.inv(V)[k1]).real   # v_1 u_1^T
print(f"|spectral P(2) - expm P(2)|_max = {np.abs(P2_spec - transition_matrix(Q3, 2.0)).max():.1e},  v_1 u_1^T row = {rank1[0].round(6)} (= pi)")

2상태의 TV는 점과 선이 정확히 겹치는 직선(기울기 $-3$)이다. $Q_3$의 곡선은 평균적으로 기울기 $-2.75$의 참조선을 따르되 주기 $2\pi/0.661\approx9.5$의 느린 진동이 얹혀 있어, 특정 구간의 기울기(예: $t=4\to6$에서 $\approx-2.92$)는 $\mathrm{Re}\lambda_2$와 다르다 (함정 4). 마지막 셀은 정리 4의 분해 $P(t)=\sum_ke^{\lambda_kt}v_ku_k^\top$가 `expm`과 같은 행렬을 주고, $\lambda=0$ 항이 정확히 $\mathbf 1\pi$임을 확인한다.

## 5. Compare — 예측 · 시뮬레이션 · 닫힌 형식 비교

In [ ]:
p00_exact, p01_exact, p02_exact = transition_matrix(Q3, 1.0)[0]   # expm 참조값

rows = [
    {"name": "P_02(1): down at t=1 from up", "predicted": predictions["p02_at_1"], "estimate": est_p2, "exact": p02_exact},
    {"name": "P_00(1): up at t=1 from up", "predicted": predictions["p00_at_1"], "estimate": est_p0, "exact": p00_exact},
    {"name": "P_01(1): degraded at t=1 from up", "predicted": None, "estimate": est_p1, "exact": p01_exact},
    {"name": "TV(P_0.(2), pi), Q3  [expm vs spectral]", "predicted": predictions["tv_at_2"], "estimate": tv3_at_2, "exact": tv3_at_2_spec},
    {"name": "TV(P_0.(1), pi), Q2  [expm vs (1/3)e^-3]", "predicted": predictions["tv_two_state_at_1"], "estimate": tv2_at_1, "exact": tv2_at_1_exact},
    {"name": "Re lambda_2(Q3)  [eigvals vs char. poly]", "predicted": predictions["re_lambda2_Q3"], "estimate": re_lambda2, "exact": re_lambda2_char},
]
Markdown(compare_table(rows))

행별로 '불일치'가 뜻하는 것:
- **P_02(1), P_00(1), P_01(1)**: 질레스피(알람 시계 구성)와 $e^{Qt}$(콜모고로프 방정식의 해)가 다른 과정을 기술한다는 뜻 — 정리 2 또는 03a 정리 3 중 하나가 틀렸거나, 격자 인덱스 `idx`가 $t=1$이 아니다. 세 값의 합은 항상 1이므로 하나가 어긋나면 다른 것도 어긋난다.
- **TV(2), Q3**: `expm`과 고윳값 분해 $Ve^{\Lambda t}V^{-1}$가 다르면 정리 4의 분해(또는 `np.linalg.eig`의 조건수)를 의심한다.
- **TV(1), Q2**: 닫힌 형식 $\pi_1e^{-(\alpha+\beta)t}$가 어긋나면 03a 정리 2의 2상태 공식이나 `tv_distance`의 $\frac12$ 규약을 확인한다.
- **Re λ₂**: 특성다항식 $\lambda^2+5.5\lambda+8$의 근의 실수부 $-5.5/2$와 `eigvals`가 다르면 $Q_3$를 잘못 입력한 것이다 (대각합 $=-5.5=\lambda_2+\lambda_3$).

In [ ]:
assert_close(est_p2, p02_exact, k=4, name="p02_at_1")
assert_close(est_p0, p00_exact, k=4, name="p00_at_1")
assert_close(est_p1, p01_exact, k=4, name="p01_at_1")
print("all Monte Carlo checks passed (4 SE)")

## 6. 연습문제

### E1 계산

손으로:
(a) 2상태 사슬($\alpha=1,\beta=2$)의 전진 방정식을 성분별로 쓰고 $P_{00}(t)$를 풀어 $P_{00}(1)$을 구하라.
(b) $Q_3$에 대해 $P'(0)=Q$임을 이용해 $P'_{01}(0)$, $P'_{00}(0)$의 값을 말하고, 근사 $P_{01}(h)\approx h$가 $h=0.01$에서 `expm` 값과 몇 자리까지 맞을지 예상하라 (힌트: 2차 항은 $(Q^2)_{01}h^2/2$).

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 전진 방정식 $P'=PQ$의 $(0,0)$ 성분: $P_{00}'=P_{00}q_{00}+P_{01}q_{10}=-\alpha P_{00}+\beta P_{01}$. $P_{01}=1-P_{00}$을 넣으면 $P_{00}'=-P_{00}+2(1-P_{00})=2-3P_{00}$, $P_{00}(0)=1$. 선형 1계 ODE의 해는 $P_{00}(t)=\frac23+\frac13e^{-3t}$ (평형값 $2/3$ + 감쇠 $e^{-(\alpha+\beta)t}$). 따라서 $P_{00}(1)=\frac23+\frac13e^{-3}\approx0.683262$. (후진 방정식 $P'_{00}=q_{00}P_{00}+q_{01}P_{10}=-P_{00}+P_{10}$은 $P_{10}$이 섞여 혼자서는 닫히지 않는다 — 두 방정식은 해가 같지만 '풀기 쉬운 쪽'이 다르다.)

(b) $P'(0)=Q$이므로 $P'_{01}(0)=q_{01}=1$, $P'_{00}(0)=q_{00}=-1.5$. 2차 항: $(Q^2)_{01}=\sum_kq_{0k}q_{k1}=(-1.5)(1)+(1)(-2)+(0.5)(0)=-3.5$이므로 $P_{01}(0.01)\approx0.01-3.5\cdot\frac{10^{-4}}{2}=0.009825$; `expm` 값은 $0.0098269$로 1차 근사 $0.01$은 소수 둘째 자리까지만 맞고 상대오차 약 1.7%다.

검산 코드:

```python
print(2/3 + np.exp(-3)/3, transition_matrix(Q2, 1.0)[0, 0])          # 0.683262 둘 다
print(transition_matrix(Q3, 1e-6)[0, 1] / 1e-6, (Q3 @ Q3)[0, 1])      # ≈ 1.0, -3.5
print(transition_matrix(Q3, 0.01)[0, 1], 0.01 - 3.5 * 0.01**2 / 2)   # 0.0098269 vs 0.009825
```

</details>

### E2 유도 후 시뮬레이션 검증

기계가 정상(0)에서 출발해 시각 1까지 **한 번도** 고장(2)이 나지 않을 확률 $P(\tau_2>1\mid X_0=0)$을 구하라. 힌트: 상태 2를 흡수 상태로 만든 $\tilde Q$ (2행을 0으로)에 대해 $e^{\tilde Qt}$의 0행에서 0, 1열의 합이 답이다 — 후진 방정식이 '2에 닿기 전' 확률을 그대로 만족한다. `gillespie_paths(Q3, 0, 1.0, rng, 8000, n_grid=1001)`로 상태 2 방문 여부(`(X == 2).any(1)` 또는 `markov.hitting_times`)를 세어 검증하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

**유도.** $\tilde Q$는 상태 2에서 나가는 알람을 모두 꺼서 2를 흡수 상태로 만든 생성원이다. 2에 처음 닿기 전까지 두 사슬(원래 $Q_3$와 $\tilde Q$)의 경로 분포는 같으므로 (같은 시계, 같은 점프 확률) $\{X_s\ne2\ \forall s\le t\}$의 확률은 $\tilde X_t\ne2$의 확률과 같다:
$$P(\tau_2>t\mid X_0=0)=[e^{\tilde Qt}]_{00}+[e^{\tilde Qt}]_{01}=1-[e^{\tilde Qt}]_{02}.$$
후진 방정식으로 보면: $u_i(t)=P_i(\tau_2>t)$는 $u'=\tilde Qu$, $u(0)=(1,1,0)$의 해이고 $\tilde Q$의 2행이 0이므로 $u_2\equiv0$이 자동으로 유지된다. $t=1$: $\approx0.532339$.

**시뮬레이션.** 격자 간격 $0.001$이면 상태 2에 머무는 시간(평균 $1/2$)이 격자 사이에 완전히 들어갈 확률은 $\approx 2\times0.001$로 무시 가능하다; 정확히 하려면 `gillespie`의 점프 시각·상태로 직접 세면 된다.

```python
Q_kill = Q3.copy(); Q_kill[2, :] = 0.0                 # 상태 2 를 흡수 상태로
p_no_fail = transition_matrix(Q_kill, 1.0)[0, :2].sum()
t_e2, X_e2 = gillespie_paths(Q3, 0, 1.0, rng, 8000, n_grid=1001)
est_no_fail = mc_proportion(~(X_e2 == 2).any(axis=1))
print(f"exact {p_no_fail:.6f}   Gillespie {est_no_fail}")
assert_close(est_no_fail, p_no_fail, k=4, name="E2 no failure by t=1")
```

출력 예: `exact 0.532339   Gillespie 0.5236 ± 0.0056 (n=8000)` — 1.6 SE 이내.

</details>

### E3 가정을 깨보기

(a) $Q_3$의 $Q_{00}$을 $-1.5$ 대신 $-1.0$으로 바꾼 $Q_{bad}$(행합 $\ne0$)에 대해 `expm(Q_bad)`의 행합을 계산하라. 무엇이 깨지는가?
(b) 03a E3의 준마르코프(semi-Markov) 2상태 과정 — 체류시간이 $\mathrm{Exp}(q_i)$ 대신 **상수** $1/q_i$ ($q=(1,2)$) — 에 대해 시뮬레이션으로 $\hat P(1)$, $\hat P(2)$를 추정하고 $\hat P(1)^2\ne\hat P(2)$ (반군 성질 실패)를 보여라. 체류를 $\mathrm{Uniform}(0,2/q_i)$로 바꿔도 같은 결론인지 확인하라.

In [ ]:
# 여기에 풀이

<details><summary>정답</summary>

(a) 행합이 $(1.401,\ 1.173,\ 1.233)$으로 1을 벗어난다. $e^{Qt}\mathbf 1=\mathbf 1$은 $Q\mathbf 1=0$에서 오므로 (급수의 $n\ge1$ 항이 모두 $\mathbf 1$을 죽인다), 행합 0 조건이 곧 확률 보존이다. $Q_{bad}$의 0행은 '탈출 비율 1.5인데 대각은 $-1$'이라 질량이 0.5의 비율로 새로 생겨난다 — 확률 과정이 아니다.

```python
Q_bad = Q3.copy(); Q_bad[0, 0] = -1.0
print(is_generator(Q_bad), expm(Q_bad).sum(axis=1))    # False [1.401066 1.173215 1.232788]
```

(b) 상수 체류이면 경로가 **결정론적**이다: 0에서 출발하면 $X_t=0$ on $[0,1)$, $1$ on $[1,1.5)$, $0$ on $[1.5,2.5)$, …; 1에서 출발하면 $X_t=1$ on $[0,0.5)$, $0$ on $[0.5,1.5)$, $1$ on $[1.5,2)$, $0$ on $[2,3)$, …. 그러면 (우연속 규약으로) $P(1)=\begin{pmatrix}0&1\\1&0\end{pmatrix}$, $P(2)=\begin{pmatrix}1&0\\1&0\end{pmatrix}$이고 $P(1)^2=I\ne P(2)$. 균등 체류에서는 원소가 0/1은 아니지만 여전히 $\hat P(1)^2\ne\hat P(2)$ (차이 $\approx0.15$, SE $\approx0.01$). 깨진 가정: 무기억성 → 마르코프 성질 → 정리 1의 둘째 등호 $P(X_{s+t}=j\mid X_s=k,X_0=i)=P_{kj}(t)$. 시각 $s$의 상태만으로는 '지금 체류가 얼마나 진행됐는지'를 알 수 없기 때문이다. (03c/04c 예고: 그래도 장기 점유율은 평균 체류시간만으로 정해져 $(2/3,1/3)$으로 같다.)

```python
def semi_markov_paths(hold, T, n_paths, rng, x0, n_grid=201):
    """2상태 교대(0->1->0->...) 준마르코프 과정; hold(x, rng) 가 상태 x 의 체류시간을 준다."""
    t_grid = np.linspace(0, T, n_grid)
    X = np.empty((n_paths, n_grid), dtype=int)
    for i in range(n_paths):
        t, x, times, states = 0.0, x0, [0.0], [x0]
        while t <= T:
            t += hold(x, rng); x = 1 - x
            times.append(t); states.append(x)
        X[i] = np.asarray(states)[np.searchsorted(times, t_grid, side="right") - 1]
    return t_grid, X

q = np.array([1.0, 2.0])
const_hold = lambda x, rng: 1 / q[x]                 # 상수 체류 (03a E3 (i))
unif_hold = lambda x, rng: rng.uniform(0, 2 / q[x])  # 평균은 같은 균등 체류 (03a E3 (ii))

def P_hat(hold, t, n_paths=2000):
    P = np.empty((2, 2))
    for x0 in (0, 1):
        _, Xs = semi_markov_paths(hold, t, n_paths, rng, x0, n_grid=2)   # 마지막 열 = X_t
        P[x0] = [(Xs[:, -1] == 0).mean(), (Xs[:, -1] == 1).mean()]
    return P

for name, hold in (("constant", const_hold), ("uniform", unif_hold)):
    P1h, P2h = P_hat(hold, 1.0), P_hat(hold, 2.0)
    print(f"{name}: P(1)^2 =\n{(P1h @ P1h).round(3)}\n P(2) =\n{P2h.round(3)}")
P1m, P2m = transition_matrix(Q2, 1.0), transition_matrix(Q2, 2.0)
print("Markov (Q2): |P(1)^2 - P(2)| =", np.abs(P1m @ P1m - P2m).max())     # 0
```

</details>

## 7. 정리

1. 채프먼–콜모고로프 $P(s+t)=P(s)P(t)$를 미분하면 전진 $P'=PQ$, 후진 $P'=QP$가 나오고, 유한 상태에서 둘의 유일한 해는 $P(t)=e^{Qt}$다.
2. 행렬 지수는 `scipy.linalg.expm`; 원소별 `np.exp`가 아니다. $e^{Qt}=\lim(I+Qt/n)^n$은 격자 근사의 극한이자 균일화(03f)의 출발점이다.
3. 질레스피 점유율은 $e^{Qt}$의 행과 몬테카를로 오차 안에서 일치한다 — 시뮬레이션과 해석해가 서로를 검증한다.
4. 기약 유한 CTMC는 주기 조건 없이 $P(t)\to\mathbf 1\pi$로 수렴한다; 속도는 $\mathrm{Re}\lambda_2$, 복소 고윳값이면 진동한다.
5. $\pi$는 $\pi Q=0$으로 계산한다(03c) — $P(t)$를 큰 $t$에서 계산할 필요가 없다.

**trap 카드**
- Q: 정상분포가 있는 유한 CTMC가 그 정상분포로 수렴하지 않을 수 있는가? (DTMC의 주기적 반례처럼)
- A: 기약이면 불가능하다. $t>0$에서 $P(t)$의 모든 원소가 양수라 주기 현상이 없고, 균일화 사슬 $I+Q/\lambda$ ($\lambda>\max q_i$)가 자기 루프를 가져 비주기적이기 때문이다. 수렴이 실패하는 경우는 기약이 아닐 때(닫힌 집합이 둘 이상)뿐이다.

**다음 노트북**: 03c — $P(t)\to\mathbf 1\pi$의 극한 $\pi$를 $e^{Qt}$ 없이 $\pi Q=0$으로 바로 구하는 법과, 출생-사망 구조에서 곱 형태의 닫힌 식이 나오는 것을 본다.

log/progress.md 한 줄 템플릿:
`- [ ] 03b 콜모고로프 방정식과 행렬 지수 — 날짜: ____ / 예측 적중: __/5 / E1 __ E2 __ E3 __ / 메모: ____`